## CitiBike Data Analysis Working Code for Charts 

### Import Libraries 

In [36]:
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from datetime import datetime as dt
from streamlit_keplergl import keplergl_static

### Wrangle data

In [37]:
df = pd.read_csv('df_temp_cleaned.csv', index_col = 0)

C:\Users\beaac\AppData\Local\Temp\ipykernel_23580\2494770719.py:1: DtypeWarning:

Columns (5,7) have mixed types. Specify dtype option on import or set low_memory=False.



In [38]:
df.dtypes

rideable_type          object
started_at             object
ended_at               object
start_station_name     object
start_station_id       object
end_station_name       object
end_station_id         object
start_lat             float64
start_lng             float64
end_lat               float64
end_lng               float64
member_casual          object
start_time             object
avgTemp               float64
_merge                 object
value                   int64
bike_rides_daily        int64
merge_flag2            object
dtype: object

In [39]:
print(df.index[:5])
print(type(df.index))


Index(['D877131AEF510B73', '0C957B74F94BDA8E', 'F5521E4F44B2F22B',
       'FDD012FF8BAA1A88', 'DF5876D1A05F8A0D'],
      dtype='object', name='ride_id')
<class 'pandas.core.indexes.base.Index'>


In [40]:
df.reset_index(inplace=True)


In [41]:
df['start_time'] = pd.to_datetime(df['start_time'])


In [42]:
df['start_station_id'] = pd.to_numeric(df['start_station_id'], errors='coerce')
df['end_station_id'] = pd.to_numeric(df['end_station_id'], errors='coerce')


In [43]:
#Convert rideable_type and member_casual to category
df['rideable_type'] = df['rideable_type'].astype('category')
df['member_casual'] = df['member_casual'].astype('category')

In [44]:
# Convert station names to category
df['start_station_name'] = df['start_station_name'].astype('category')
df['end_station_name'] = df['end_station_name'].astype('category')

In [45]:

# Convert merge flags to category
df['_merge'] = df['_merge'].astype('category')
df['merge_flag2'] = df['merge_flag2'].astype('category')

## Bar chart: Most Popular Stations in New York

In [46]:
df['value'] = 1

In [47]:
df_groupby_bar = df.groupby('start_station_name', as_index=False).agg({'value' : 'sum'})
top20 = df_groupby_bar.nlargest(20, 'value')

In [ ]:
## Bar chart
fig = go.Figure(go.Bar(x = top20['start_station_name'], y = top20['value'], marker={'color': top20['value'],'colorscale': 'Blues'}))
fig.show()



In [ ]:
# Bar Chart
fig.update_layout(
    title = 'Top 20 most popular bike stations in New York',
    xaxis_title = 'Start stations',
    yaxis_title ='Sum of trips',
    width = 900, height = 600
)

## Dual Axis Line Chart: Bike Trips and Temperature


In [50]:
df.set_index('start_time', inplace=True)


In [51]:
df_daily = df.resample('D').mean()


C:\Users\beaac\AppData\Local\Temp\ipykernel_23580\1664659276.py:1: FutureWarning:

The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.



In [ ]:
 fig_2 = make_subplots(specs = [[{"secondary_y": True}]])

fig_2.add_trace(
    go.Scatter(
        x=df_daily.index,
        y=df_daily['bike_rides_daily'],
        name='Daily bike rides',
        line=dict(color='blue')
    ),
    secondary_y=False
)

fig_2.add_trace(
    go.Scatter(
        x=df_daily.index,
        y=df_daily['avgTemp'],
        name='Daily temperature',
        line=dict(color='red')
    ),
    secondary_y=True
)
fig_2.update_layout(
    title='Daily Bike Trips and Temperatures in 2022',
    height=600,
)





In [53]:
# Save the top 20 stations as a csv file 

top20.to_csv('top20.csv')

In [72]:
# save the df_daily

df_daily.to_csv('df_dashboard_ready.csv')

## Reduce the row and column count

In [61]:
print(df.index[:5])
print(type(df.index))


DatetimeIndex(['2022-01-01 00:52:35.165000', '2022-01-01 03:16:55.870000',
               '2022-01-01 02:54:36.998000', '2022-01-01 00:17:39.433000',
               '2022-01-01 00:36:35.058000'],
              dtype='datetime64[ns]', name='start_time', freq=None)
<class 'pandas.core.indexes.datetimes.DatetimeIndex'>


## Create a random split

In [4]:
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from datetime import datetime as dt
from streamlit_keplergl import keplergl_static

In [6]:
df = pd.read_csv('df_temp_cleaned.csv', index_col = 0)

C:\Users\beaac\AppData\Local\Temp\ipykernel_4076\2494770719.py:1: DtypeWarning: Columns (5,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('df_temp_cleaned.csv', index_col = 0)


In [9]:
df.columns

Index(['rideable_type', 'started_at', 'ended_at', 'start_station_name',
       'start_station_id', 'end_station_name', 'end_station_id', 'start_lat',
       'start_lng', 'end_lat', 'end_lng', 'member_casual', 'start_time',
       'avgTemp', '_merge', 'value', 'bike_rides_daily', 'merge_flag2'],
      dtype='object')

In [10]:
# Create a copy with fewer columns

df_1 = df.drop(columns = {'rideable_type', 'started_at', 'ended_at','start_station_id', 'end_station_name', 'end_station_id', 'start_lat',
       'start_lng', 'end_lat', 'end_lng', 'member_casual', 'start_time','merge_flag2'}) 

In [11]:
np.random.seed(32)
red = np.random.rand(len(df_1)) <= 0.92

In [12]:
small = df_1[~red]

In [13]:
small.shape

(2379402, 5)

In [14]:
small.to_csv('reduced_data_to_plot_7.csv',index = False)

## Adding FacetGrid from Homework 2.4

In [2]:
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from datetime import datetime as dt
from streamlit_keplergl import keplergl_static

In [5]:
df_1 = pd.read_csv('df_temp_cleaned.csv', index_col = 0)

C:\Users\beaac\AppData\Local\Temp\ipykernel_29932\1552926890.py:1: DtypeWarning: Columns (5,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_1 = pd.read_csv('df_temp_cleaned.csv', index_col = 0)


In [6]:
df_1.columns


Index(['rideable_type', 'started_at', 'ended_at', 'start_station_name',
       'start_station_id', 'end_station_name', 'end_station_id', 'start_lat',
       'start_lng', 'end_lat', 'end_lng', 'member_casual', 'start_time',
       'avgTemp', '_merge', 'value', 'bike_rides_daily', 'merge_flag2'],
      dtype='object')

In [8]:
# Convert timestamps to datetime
df_1['started_at'] = pd.to_datetime(df_1['started_at'])
df_1['ended_at'] = pd.to_datetime(df_1['ended_at'])

In [9]:
# Calculate ride length in minutes
df_1['ride_length'] = (df_1['ended_at'] - df_1['started_at']).dt.total_seconds() / 60

In [10]:
# Drop timestamp columns
df_1.drop(columns=['started_at', 'ended_at'], inplace=True)

In [11]:
# Optional: remove negative or extreme durations
df_1 = df_1[df_1['ride_length'] > 0]
df_1 = df_1[df_1['ride_length'] < 180]  # under 3 hours

In [12]:
df_1 = df_1[['member_casual', 'ride_length']]


In [20]:
df_1.columns

Index(['member_casual', 'ride_length'], dtype='object')

In [ ]:
# Set Seaborn style
import seaborn as sns
sns.set(style="whitegrid")

# Create FacetGrid
g = sns.FacetGrid(df_1, col='member_casual', height=5, aspect=1.2)
g.map(sns.histplot, 'ride_length', bins=30, color='steelblue')

# Label axes and titles
g.set_axis_labels("Ride Duration (minutes)", "Frequency")
g.set_titles("{col_name} Users")
g.fig.suptitle("Distribution of Ride Duration by User Type", fontsize=16)

# Adjust layout
plt.tight_layout()
plt.subplots_adjust(top=0.85)
plt.show()

In [21]:
# Save trimmed dataset
df_1.to_csv('df_rides_trimmed.csv')

In [18]:
df_2 = pd.read_csv('df_rides_trimmed.csv', index_col = 0)

In [19]:
df_2.columns

Index(['ride_length'], dtype='object')